In [1]:
import requests as rq
import psycopg2 as p2
from datetime import datetime, timedelta
import pandas as pd
from pprint import pprint
import os
from dotenv import load_dotenv
from io import StringIO
from concurrent.futures import ThreadPoolExecutor
import requests as rq
import time
import threading

##### Знакомство с API

In [2]:
# выясним, как работает API
api_url = 'http://final-project.simulative.ru/data'
params = {'date': 'yyyy-mm-dd'}
response = rq.get(api_url, params=params)
pprint(response.json())

[{'client_id': 615487,
  'discount_per_item': 71831,
  'gender': 'F',
  'price_per_item': 81947,
  'product_id': 46801,
  'purchase_datetime': 'yyyy-mm-dd',
  'purchase_time_as_seconds_from_midnight': 74605,
  'quantity': 9,
  'total_price': 91044},
 {'client_id': 58508,
  'discount_per_item': 4102,
  'gender': 'F',
  'price_per_item': 31176,
  'product_id': 42859,
  'purchase_datetime': 'yyyy-mm-dd',
  'purchase_time_as_seconds_from_midnight': 26455,
  'quantity': 57,
  'total_price': 1543218},
 {'client_id': 225771,
  'discount_per_item': 20922,
  'gender': 'M',
  'price_per_item': 26215,
  'product_id': 12226,
  'purchase_datetime': 'yyyy-mm-dd',
  'purchase_time_as_seconds_from_midnight': 40442,
  'quantity': 40,
  'total_price': 211720},
 {'client_id': 17281,
  'discount_per_item': 20573,
  'gender': 'F',
  'price_per_item': 22635,
  'product_id': 14793,
  'purchase_datetime': 'yyyy-mm-dd',
  'purchase_time_as_seconds_from_midnight': 20298,
  'quantity': 63,
  'total_price': 12990

In [3]:
# видим, что API выдает данные с yyyy-mm-dd вместо даты
# посмотрим на длину полученного json
len(response.json())

1417

In [4]:
# теперь выполним точно такой же запрос несколько раз и посмотрим на длину json
for i in range(4):
    response = rq.get(api_url, params=params)
    print(len(response.json()))
# очевидно, перед нами генератор данных, который подставляет предоставленный параметр date  
# значит, цель - прописать скрипт, который сгенерирует данные за выбранный промежуток времени

7172
6335
3109
8120


In [5]:
# задание - проанализировать данные за 2025 год, загрузим данные с начала 2025
params = {'date': '2025-01-01'}
try:
    response = rq.get(api_url, params=params)
    pprint(response.json()[0])
except:
    print("Данные за этот день отсутствуют.")
# данные есть

{'client_id': 123885,
 'discount_per_item': 3146,
 'gender': 'M',
 'price_per_item': 83857,
 'product_id': 10268,
 'purchase_datetime': '2025-01-01',
 'purchase_time_as_seconds_from_midnight': 63390,
 'quantity': 47,
 'total_price': 3793417}


##### Заполнение базы историческими данными

In [6]:
# генерируем список интересующих нас дат
start_date = datetime.fromisoformat('2025-01-01').date()
end_date = datetime.date(datetime.today() - timedelta(days=1))

date_list = [start_date + timedelta(days=i) for i in range((end_date - start_date).days + 1)]

# трансформируем в формат 'yyyy-mm-dd'
date_list_for_script = [f'{i.year}-{i.month:02d}-{i.day:02d}' for i in date_list]

In [7]:
# получаем данные для подключения к БД и настраиваем подключение
result = load_dotenv()
if not result:
    raise EnvironmentError('Не удалось загрузить файл .env или переменные окружения')

db_host = os.getenv('DB_HOST')
db_name = os.getenv('DB_NAME')
db_user = os.getenv('DB_USER')
db_pass = os.getenv('DB_PASS')
db_port = os.getenv('DB_PORT')

conn = p2.connect(database=db_name, user=db_user, password=db_pass, host=db_host, port=db_port)

#conn.autocommit = True
cursor = conn.cursor()
conn.status

1

In [8]:
# тест добавления информации
data_for_db = []
response = rq.get(api_url, params={'date': date_list_for_script[0]})
data_for_db.extend(response.json())
data_for_db

[{'client_id': 351268,
  'gender': 'M',
  'purchase_datetime': '2025-01-01',
  'purchase_time_as_seconds_from_midnight': 31484,
  'product_id': 36586,
  'quantity': 20,
  'price_per_item': 63770,
  'discount_per_item': 39402,
  'total_price': 487360},
 {'client_id': 997338,
  'gender': 'F',
  'purchase_datetime': '2025-01-01',
  'purchase_time_as_seconds_from_midnight': 34658,
  'product_id': 8793,
  'quantity': 11,
  'price_per_item': 84505,
  'discount_per_item': 45253,
  'total_price': 431772},
 {'client_id': 355667,
  'gender': 'M',
  'purchase_datetime': '2025-01-01',
  'purchase_time_as_seconds_from_midnight': 82993,
  'product_id': 37373,
  'quantity': 21,
  'price_per_item': 85546,
  'discount_per_item': 42517,
  'total_price': 903609},
 {'client_id': 813517,
  'gender': 'F',
  'purchase_datetime': '2025-01-01',
  'purchase_time_as_seconds_from_midnight': 12934,
  'product_id': 49474,
  'quantity': 11,
  'price_per_item': 91192,
  'discount_per_item': 28737,
  'total_price': 68

In [9]:
# формируем базу данных со всеми историческими данными
# общая сессия для всех потоков
import requests as rq
import time

# изолированное хранилище для каждого потока
thread_local = threading.local()

def get_session():
    if not hasattr(thread_local, "session"):
        thread_local.session = rq.Session()
    return thread_local.session

def fetch_single_date(date, max_retries=3):
    session = get_session()
    
    for attempt in range(max_retries):
        try:
            response = session.get(api_url, params={'date': date}, timeout=25)
            
            if response.status_code == 200:
                return response.json()
            elif response.status_code == 429:
                # Сервер перегружен: пауза растет с каждой попыткой
                time.sleep(2 ** (attempt + 1))
            else:
                # Ошибки 4xx/5xx
                time.sleep(1)
                
        except rq.exceptions.RequestException:
            # Сетевой сбой или таймаут: прогрессивная пауза (1с, 2с, 4с...)
            time.sleep(1.5 ** attempt)
            
    print(f"Не удалось получить данные за {date} после {max_retries} попыток.")
    return []

data_for_db = []
total = len(date_list_for_script)
completed = 0
start_time = time.time()

print(f"Старт сбора данных: {total} дат в 3 потока...")

with ThreadPoolExecutor(max_workers=3) as executor:
    for chunk in executor.map(fetch_single_date, date_list_for_script):
        if chunk:
            data_for_db.extend(chunk)
            
        completed += 1
        if completed % 50 == 0 or completed == total:
            elapsed = time.time() - start_time
            print(f"Обработано {completed}/{total} дат ({elapsed:.1f} сек)...")

print(f"\nСбор завершен! Записей: {len(data_for_db)} за {(time.time() - start_time) / 60:.1f} мин")

Старт сбора данных: 670 дат в 3 потока...
Обработано 50/670 дат (45.0 сек)...
Обработано 100/670 дат (92.4 сек)...
Обработано 150/670 дат (140.6 сек)...
Обработано 200/670 дат (185.6 сек)...
Обработано 250/670 дат (230.2 сек)...
Обработано 300/670 дат (274.0 сек)...
Обработано 350/670 дат (318.3 сек)...
Обработано 400/670 дат (361.6 сек)...
Обработано 450/670 дат (402.3 сек)...
Обработано 500/670 дат (444.1 сек)...
Обработано 550/670 дат (491.8 сек)...
Обработано 600/670 дат (538.1 сек)...
Обработано 650/670 дат (581.0 сек)...
Обработано 670/670 дат (600.2 сек)...

Сбор завершен! Записей: 3703026 за 10.0 мин


In [ ]:
# теперь задача - загрузить это в БД PostgreSQL
# создаем таблицу sales в БД, задаем типы данных (не создаем столбец purchase_time_as_seconds_from_midnight, 
# потому что данные будут интегрированы в столбец purchase_datetime)
query = """create table if not exists sales (
    id serial primary key,
    client_id bigint,
    gender char(1),
    purchase_datetime timestamp,
    product_id bigint,
    quantity integer,
    price_per_item numeric(10, 2),
    discount_per_item numeric(10, 2),
    total_price numeric(12, 2)
)"""

try:
    cursor.execute(query)
    conn.commit()
except Exception as err:
    raise repr(err)

In [ ]:
# воспользуемся методом .copy_from для загрузки всех данных одним запросом к БД
upload_df = pd.DataFrame(data_for_db)

upload_df['purchase_datetime'] = pd.to_datetime(upload_df['purchase_datetime']) + \
                                 pd.to_timedelta(upload_df['purchase_time_as_seconds_from_midnight'], unit='s')

def upload_to_db(df, table_name):
    df = df.drop(columns=['purchase_time_as_seconds_from_midnight'])
    output = StringIO()
    df.to_csv(output, sep='\t', header=False, index=False)
    output.seek(0)
    columns = (
        'client_id', 'gender', 'purchase_datetime', 'product_id', 
        'quantity', 'price_per_item', 'discount_per_item', 'total_price'
    )
    cursor.copy_from(output, table_name, sep='\t', columns=columns)
    conn.commit()

upload_to_db(upload_df, 'sales')

In [12]:
# проверим, успешно ли загрузились данные
query = """
    select client_id, purchase_datetime, total_price
    from sales
    order by purchase_datetime desc, total_price desc
    limit 5
"""
cursor.execute(query)
cursor.fetchall()
# данные загружены успешно

[(960149, datetime.datetime(2026, 11, 1, 23, 59, 45), Decimal('978965.00')),
 (639898, datetime.datetime(2026, 11, 1, 23, 59, 34), Decimal('35838.00')),
 (132970, datetime.datetime(2026, 11, 1, 23, 59, 24), Decimal('478669.00')),
 (760197, datetime.datetime(2026, 11, 1, 23, 59, 19), Decimal('2068416.00')),
 (788763, datetime.datetime(2026, 11, 1, 23, 58, 59), Decimal('386627.00'))]

In [13]:
conn.close()

In [14]:
conn.closed

1